## Log and register model: native scikit-learn

### Purpose
This notebook ingests a native scikit-learn model trained outside Databricks,
logs it to MLflow, and registers it in Unity Catalog for governed deployment.

Use this notebook when:
- The customer provides a serialized scikit-learn model artifact (e.g. joblib/pickle)
- No custom PyFunc wrapper is required
- You want to preserve the native MLflow scikit-learn flavour

This is Stage 2 of the BYOM workflow (Log & Register).

### Expected inputs
Artifacts must already exist in a Unity Catalog volume and follow the expected layout defined in Stage 1.

Required:
- Serialized scikit-learn model file (e.g. `sklearn_model.pkl`)

Optional but recommended:
- `canonical_input.json`
- `checksums.json` for artifact integrity validation


In [ ]:
%pip install -q threadpoolctl>=3.5.0 xgboost torch
%pip install --upgrade 'mlflow>=3.0.0'
%restart_python

In [ ]:
dbutils.widgets.text("catalog_name", "pcl", "Catalog Name")
dbutils.widgets.text("schema_name", "byo_model", "Schema Name")
dbutils.widgets.text("volume_name", "volume", "Volume Name")

catalog = dbutils.widgets.get("catalog_name")
schema = dbutils.widgets.get("schema_name")
volume = dbutils.widgets.get("volume_name")

### Load Artifacts from Unity Catalog Volume

Reads model artifacts from the configured UC volume using notebook widgets:

- `catalog_name`
- `schema_name`
- `volume_name`

Ensure these are aligned before running.

If `checksums.json` is present:
- Verify artifact integrity before logging
- Fail fast if mismatches are detected

Recommended for production engagements.


In [ ]:
%run ./00_shared

In [ ]:
import joblib
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

# --- Paths and verification ---
model_name = "native_sklearn"
artifact_root = get_artifact_root(catalog, schema, volume)
cfg = get_native_sklearn_paths(artifact_root)
checksums_path = get_checksums_path(artifact_root)
verify_artifacts(checksums_path, [(cfg["checksum_key"], cfg["path"])])
canonical_input_path = get_canonical_input_path(artifact_root)
verify_artifacts(checksums_path, [("canonical_input.json", canonical_input_path)])
canonical_input, feature_columns = load_canonical_input_json(canonical_input_path)


### Infer model signature

Infers input/output schema using canonical sample input (if provided).  
The inferred signature ensures consistent contract enforcement during serving and batch inference.

In [ ]:
# --- Load model, run one prediction, infer MLflow signature ---
model = joblib.load(cfg["path"])
y_pred = pd.DataFrame({"prediction": model.predict(canonical_input.values)})
signature = validate_and_infer_signature(canonical_input, y_pred)

### Log to MLflow and register to Unity Catalog

Logs the native scikit-learn model using `mlflow.sklearn`. Artifacts, signature, and metadata are recorded in MLflow tracking.

Registers the logged model in Unity Catalog Model Registry.

Ensures:
- Governance
- Lineage
- Versioning

In [ ]:
# --- Log to MLflow using native sklearn flavor, then register in Unity Catalog ---
mlflow.set_registry_uri("databricks-uc")
deps = ["scikit-learn", "pandas", "numpy"]
with mlflow.start_run() as run:
    mlflow.sklearn.log_model(model, artifact_path=model_name, input_example=canonical_input, signature=signature, extra_pip_requirements=deps)
    model_uri = f"runs:/{run.info.run_id}/{model_name}"
registered_name = f"{catalog}.{schema}.{model_name}"
result = mlflow.register_model(model_uri=model_uri, name=registered_name)
client = MlflowClient()
register_in_uc(client, model_uri, registered_name, str(result.version))

# For job pipelines: pass version to downstream tasks (e.g. evaluation)
dbutils.jobs.taskValues.set(key="model_version", value=str(result.version))
dbutils.jobs.taskValues.set(key="model_name", value=registered_name)

# Sanity check: load the version we just registered and run predict (Champion is set later in 3_model_approval)
loaded = mlflow.pyfunc.load_model(f"models:/{registered_name}/{result.version}")
loaded.predict(canonical_input)
print(f"{registered_name} v{result.version} loaded; predict OK.")